# บทเรียนที่ 18: การรักษาความปลอดภัยของเอเจนต์ AI ด้วยใบเสร็จทางคริปโตกราฟี

## โน้ตบุ๊กฝึกปฏิบัติ

โน้ตบุ๊กนี้นำทางผ่านสี่แบบฝึกหัด:

1. **ลงนามใบเสร็จแรกของคุณ** สำหรับการเรียกใช้เครื่องมือของเอเจนต์และตรวจสอบความถูกต้อง
2. **ปลอมแปลงใบเสร็จ** และดูว่าการตรวจสอบล้มเหลว
3. **สร้างโซ่ใบเสร็จสามใบ** และยืนยันความสมบูรณ์ของโซ่
4. **ห่อหุ้มการเรียกใช้เครื่องมือ Microsoft Agent Framework** เพื่อให้ทุกการกระทำสร้างใบเสร็จ

ไพรเมทีฟคริปโตกราฟีทั้งหมดถูกนำเข้าจากไลบรารีที่ดูแลอย่างดี (`pynacl` สำหรับ Ed25519, `jcs` สำหรับ RFC 8785 canonical JSON, `hashlib` จากไลบรารีมาตรฐาน Python สำหรับ SHA-256) ตัวตรรกะใบเสร็จเองเขียนด้วย Python ธรรมดาที่คุณสามารถอ่านและแก้ไขได้

รันเซลล์ตามลำดับ แต่ละส่วนสั้นและเป็นอิสระจากกัน


## Setup

ติดตั้ง dependencies ทั้งสองตัว ทั้งคู่มีใบอนุญาตแบบ permisive (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

ตัวช่วยสองตัวนี้จัดการการเข้ารหัส base64url (โดยไม่เติม padding) และการแฮช SHA-256 ของวัตถุใดๆ พวกมันช่วยให้สมุดบันทึกที่เหลือมุ่งเน้นไปที่ตรรกะใบเสร็จเองโดยตรง


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: เซ็นชื่อใบเสร็จรับเงินใบแรกของคุณ

ลองนึกภาพตัวแทนของเราใน **Contoso Travel** เพิ่งค้นหาตั๋วเครื่องบินจากซิดนีย์ไปยังลอสแองเจลิสสำหรับลูกค้า เราต้องการบันทึกการเรียกใช้เครื่องมือนี้เป็นใบเสร็จรับเงินที่มีลายเซ็นเพื่อให้นักตรวจสอบในอนาคตสามารถตรวจสอบได้โดยไม่ต้องเชื่อใจเรา

### Step 1.1: สร้างกุญแจสำหรับเซ็นชื่อ

ในสถานการณ์จริง กุญแจสำหรับเซ็นชื่อของตัวแทนจะถูกเก็บไว้ในฮาร์ดแวร์โมดูลความปลอดภัย (HSM), Azure Key Vault หรือที่จัดเก็บที่ป้องกันในลักษณะเดียวกัน สำหรับบทเรียนนี้ เราจะสร้างกุญแจใหม่ในหน่วยความจำ


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ขั้นตอนที่ 1.2: สร้างข้อมูลการรับเงิน

ข้อมูลการรับเงินประกอบด้วยทุกอย่างที่เราต้องการให้การรับเงินยืนยัน: ใครเป็นผู้ดำเนินการ เครื่องมืออะไร พร้อมด้วยอาร์กิวเมนต์อะไร มีผลลัพธ์อะไร กลางนโยบายอะไร และเมื่อใด เราจะแฮชอาร์กิวเมนต์และผลลัพธ์แทนที่จะใส่ไว้ในบรรทัดเดียวเพื่อไม่ให้ใบเสร็จรั่วไหลข้อมูลที่ละเอียดอ่อน


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: เซ็นชื่อและประกอบใบเสร็จ

สามขั้นตอน:

1. ทำให้ payload เป็นแบบ canonical โดยใช้ JCS เพื่อให้สองการใช้งานที่สร้างใบเสร็จที่มีตรรกะเหมือนกันได้ไบต์ที่เหมือนกันทุกประการ
2. แฮชไบต์ canonical ด้วย SHA-256
3. เซ็นแฮชด้วยกุญแจส่วนตัว Ed25519

จากนั้นลายเซ็นจะถูกแนบเข้ากับ payload ดั้งเดิมเพื่อสร้างใบเสร็จขั้นสุดท้าย


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ขั้นตอนที่ 1.4: ยืนยันใบเสร็จ

การยืนยันจะย้อนกลับกระบวนการ เราจะลบลายเซ็นออก คำนวณแฮชแบบ canonical ใหม่ และตรวจสอบลายเซ็นกับกุญแจสาธารณะที่อยู่ในใบเสร็จ

ผู้ตรวจสอบที่ทำการยืนยันนี้ไม่จำเป็นต้องใช้สิ่งใดจากเรานอกจากใบเสร็จเอง ไม่มีบริการให้เรียกใช้งาน ไม่มีไดเรกทอรีคีย์ให้สอบถาม และไม่ต้องมีความเชื่อใดๆ


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

คุณควรเห็น `Receipt is valid: True` ตัวแทนได้สร้างบันทึกการตรวจสอบที่ลงนามด้วยลายเซ็นทางคริปโตกราฟีครั้งแรกแล้ว


## Section 2: ปลอมแปลงใบเสร็จ

จุดประสงค์ทั้งหมดของใบเสร็จคือเพื่อให้เห็นความพยายามในการปลอมแปลงได้ มาพิสูจน์กัน

เราจะเปลี่ยนแปลงตัวอักษรเพียงหนึ่งตัวในใบเสร็จและดูว่าการตรวจสอบจะล้มเหลวหรือไม่


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### เพิ่งเกิดอะไรขึ้น?

เมื่อเราเปลี่ยน `policy_id` ไบต์มาตรฐานถูกเปลี่ยนไป ค่าแฮช SHA-256 ของไบต์เหล่านั้นจึงเปลี่ยนไป ลายเซ็น (ซึ่งอยู่บนแฮชเดิม) จึงไม่ตรงกับแฮชใหม่อีกต่อไป การตรวจสอบจึงคืนค่า `False` อย่างถูกต้อง

ไม่มีทางแก้ไขฟิลด์ใด ๆ ของใบเสร็จและยังคงผ่านการตรวจสอบได้ เว้นแต่โจรกรรมจะมีคีย์ส่วนตัว ตราบใดที่คีย์ส่วนตัวถูกเก็บไว้ในที่เก็บคีย์และคีย์สาธารณะถูกเผยแพร่ การปลอมแปลงเป็นสิ่งที่ไม่สามารถซ่อนได้

ลองด้วยตัวเอง: แก้ไข `tool_name` หรือ `agent_id` หรือ `timestamp` ในเซลล์ข้างบนแล้วรันใหม่ การเปลี่ยนแปลงทุกครั้งจะทำให้ใบเสร็จไม่ถูกต้อง


## Section 3: การเชื่อมใบเสร็จเข้าด้วยกัน

ใบเสร็จแต่ละใบจะป้องกันการกระทำเพียงครั้งเดียว ตัวแทนส่วนใหญ่ทำหลายการกระทำ เพื่อทำให้ลำดับทั้งหมดสามารถตรวจจับการเปลี่ยนแปลงได้ เราจึงเชื่อมใบเสร็จแต่ละใบเข้ากับใบเสร็จก่อนหน้าโดยการใส่แฮชของใบเสร็จใบก่อนหน้าในข้อมูลของใบเสร็จใบใหม่

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

หากมีใครลบหรือจัดเรียงใบเสร็จใหม่ ระบบจะขัดข้องตรงจุดนั้น การยืนยันใบเสร็จในภายหลังจะล้มเหลวเพราะ `previous_receipt_hash` ไม่ตรงกับแฮชจริงของใบเสร็จที่มาก่อนหน้าอีกต่อไป


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ตอนนี้ให้ทำลายโซ่โดยการปลอมแปลงใบเสร็จตรงกลางแล้วตรวจสอบใหม่ ใบเสร็จที่ถูกปลอมแปลงจะล้มเหลวในการตรวจสอบลายเซ็น และใบเสร็จถัดไปจะล้มเหลวในการตรวจสอบการเชื่อมโยงโซ่ (เพราะค่า `previous_receipt_hash` ของมันไม่ตรงกับแฮชของใบเสร็จตรงกลางที่ถูกแก้ไข)


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ใบเสร็จ 0 ยังคงตรวจสอบได้ (ไม่ได้ถูกแก้ไขและไม่มีใบเสร็จก่อนหน้าที่ต้องพึ่งพิง) ใบเสร็จ 1 ล้มเหลวในการตรวจสอบลายเซ็นเพราะเราได้เปลี่ยน `tool_args_hash` ใบเสร็จ 2 ล้มเหลวในการตรวจสอบการเชื่อมโยงของโซ่อันเนื่องมาจาก `previous_receipt_hash` ของมันถูกคำนวณกับใบเสร็จ 1 ที่เป็นต้นฉบับ (ซึ่งตอนนี้ถูกแก้ไขแล้ว)

แม้ว่าผู้โจมตีจะลงลายเซ็นใหม่กับใบเสร็จ 1 ที่ถูกแก้ไข (ซึ่งพวกเขาไม่สามารถทำได้หากไม่มีคีย์ส่วนตัว) ความไม่ตรงกันของการเชื่อมโยงโซ่ในใบเสร็จ 2 ก็ยังคงเปิดเผยการปลอมแปลง หากต้องการปกปิดการเปลี่ยนแปลง ผู้โจมตีจะต้องลงลายเซ็นใหม่กับใบเสร็จทุกใบจากจุดที่แก้ไขเป็นต้นไป ซึ่งจำเป็นต้องมีคีย์ส่วนตัวด้วย


## Section 4: ห่อหุ้มการเรียกใช้งานเครื่องมือของเอเจนต์ด้วยการลงนามใบเสร็จ

ในการใช้งานจริง คุณไม่ต้องการให้ผู้เขียนเอเจนต์ทุกคนจำเป็นต้องเรียกใช้ `make_receipt` ด้วยตัวเอง คุณต้องการให้การลงนามใบเสร็จเป็นไปโดยอัตโนมัติสำหรับการเรียกใช้เครื่องมือแต่ละครั้ง

นี่คือลวดลายที่ง่ายที่สุด: คลาส wrapper ที่รับฟังก์ชันเครื่องมือที่สามารถเรียกใช้งานได้ใดๆ และส่งกลับเวอร์ชันที่สร้างใบเสร็จออกมา นี่ปรับใช้ได้กับเฟรมเวิร์กเอเจนต์ใดๆ รวมถึง Microsoft Agent Framework (`agent_framework.azure`)

หากคุณยังไม่มีโปรเจกต์ Azure AI Foundry ตั้งค่าไว้ ม็อกท้องถิ่นด้านล่างนี้ยังแสดงลวดลายดังกล่าวได้อยู่ดี


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### การผสานรวมกับ Microsoft Agent Framework

ตัวห่อ `ReceiptedTool` ข้างต้นเป็นอิสระจากเฟรมเวิร์ก ในการใช้งานภายในเอเจนต์ที่สร้างด้วย Microsoft Agent Framework ให้ลงทะเบียนฟังก์ชันที่ถูกห่อไว้ในฐานะเครื่องมือ โครงร่าง (คุณจะต้องแทนที่ mock ด้วยการลงทะเบียนเครื่องมือ Azure AI Foundry จริง):

```python
# รหัสปลอมแสดงรูปร่างของการรวมข้อมูล
# จาก agent_framework.azure นำเข้า AzureAIProjectAgentProvider
# จาก azure.identity นำเข้า AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     คำแนะนำ="คุณเป็นตัวแทนท่องเที่ยวของ Contoso ..."
#     tools=[receipted_lookup],   # เครื่องมือที่ถูกห่อหุ้ม ไม่ใช่ฟังก์ชันดิบ
# )
# response = agent.run("ค้นหาเที่ยวบินจากซิดนีย์ไปลอสแองเจลิสในเดือนมิถุนายน")
#
# # หลังจากการรัน ทุกการเรียกเครื่องมือที่ตัวแทนทำมีใบเสร็จลงลายมือชื่อ:
# audit_chain = receipted_lookup.receipts
```

เฟรมเวิร์กเอเจนต์ไม่จำเป็นต้องรู้เกี่ยวกับใบเสร็จใดๆ การเซ็นชื่อใบเสร็จถูกห่อหุ้มรอบเครื่องมือ ไม่ได้ผนวกเข้าไปในเฟรมเวิร์ก นี่คือวิธีที่คุณเพิ่มต้นกำเนิดให้กับโค้ดเอเจนต์ที่มีอยู่โดยไม่ต้องเขียนเอเจนต์ใหม่ทั้งหมดใหม่อีกครั้ง


## สรุปและความท้าทายเพิ่มเติม

คุณได้:

- สร้างคู่กุญแจ Ed25519 ขึ้นมา
- สร้างและลงนามใบเสร็จสำหรับการเรียกใช้งานเครื่องมือของเอเย่นต์
- ตรวจสอบใบเสร็จแบบออฟไลน์โดยใช้เพียงกุญแจสาธารณะ
- ปลอมแปลงใบเสร็จและสังเกตเห็นการตรวจสอบล้มเหลว
- สร้างลำดับใบเสร็จที่เชื่อมต่อด้วยแฮชจำนวนสามใบ
- ปลอมแปลงตรงกลางของโซ่และสังเกตเห็นทั้งการล้มเหลวของลายเซ็นและล้มเหลวของการเชื่อมโซ่
- ห่อหุ้มฟังก์ชันเครื่องมือด้วยการลงนามใบเสร็จโดยอัตโนมัติ

**ความท้าทายเพิ่มเติม.** ขยายโครงสร้างใบเสร็จด้วยฟิลด์ `request_id` (UUID สำหรับการติดตามแบบกระจาย) อัปเดต `make_receipt` ให้รวมฟิลด์นี้ และยืนยันว่าใบเสร็จยังคงตรวจสอบได้ครบถ้วน จากนั้นแก้ไขฟิลด์หลังการลงนามและยืนยันว่าการตรวจสอบล้มเหลว ซึ่งจะบังคับให้คุณทำความเข้าใจว่าทุกไบต์ของการเข้ารหัสมาตรฐานมีผลต่อการลงนามอย่างไร

**ขอบเขตสำคัญ.** ใบเสร็จพิสูจน์สิ่งสามอย่างเท่านั้นและมีเพียงสามอย่าง: การอ้างอิงความเป็นเจ้าของ (กุญแจนี้ลงนามเนื้อหานี้), ความสมบูรณ์ (เนื้อหาไม่ได้ถูกเปลี่ยนแปลงหลังการลงนาม), และการเรียงลำดับ (ใบเสร็จนี้ออกหลังใบเสร็จนั้น) พวกมันไม่ได้พิสูจน์ว่าการกระทำของเอเย่นต์ถูกต้อง นโยบายที่ระบุใน `policy_id` ได้รับการประเมินจริงหรือไม่ หรือว่าเอเย่นต์ปฏิบัติตามกฎทุกข้อหรือไม่ ใบเสร็จเป็นพื้นฐาน ระบบการปกครองคือระบบที่คุณสร้างขึ้นข้างบน

อ่าน README บทเรียนอีกครั้งโดยคำนึงถึงขอบเขตนี้ ข้อผิดพลาดที่พบบ่อยที่สุดที่ทีมทำกับใบเสร็จคือการสมมติว่า "เรามีใบเสร็จ" หมายความว่า "เราถูกปกครอง" ซึ่งไม่เป็นเช่นนั้น ใบเสร็จทำให้พฤติกรรมของเอเย่นต์ตรวจสอบได้เท่านั้น ไม่ทำให้พฤติกรรมถูกต้อง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
